# Migrazioni, saldo migratorio e popolazione nata all'estero

Questo foglio separa i concetti: il saldo migratorio appartiene al bilancio demografico, i flussi annui descrivono ingressi e uscite, mentre paese di nascita e titolo di studio descrivono la composizione della popolazione residente.

Fonti principali: [`demo_gind`](https://ec.europa.eu/eurostat/databrowser/view/demo_gind/default/table?lang=en), [`demo_r_gind3`](https://ec.europa.eu/eurostat/databrowser/view/demo_r_gind3/default/table?lang=en), [`migr_imm1ctz`](https://ec.europa.eu/eurostat/databrowser/product/view/migr_imm1ctz), [`migr_emi1ctz`](https://ec.europa.eu/eurostat/databrowser/product/view/migr_emi1ctz), [`migr_pop3ctb`](https://ec.europa.eu/eurostat/databrowser/view/migr_pop3ctb/default/table?lang=en) ed [`edat_lfs_9917`](https://ec.europa.eu/eurostat/databrowser/product/view/edat_lfs_9917). Ogni grafico riporta fonte ed elaborazione in basso a sinistra.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from demografia.notebook_charts import (
    fig_births_deaths,
    fig_internal_migration_destinations,
    fig_kebab,
    fig_migrant_education_by_birth,
    fig_migrant_tertiary_region,
    fig_migration,
    fig_migration_age_profile,
    fig_migration_citizenship_profile,
    fig_migration_destination_rank,
    fig_population_series,
    fig_regional_rank,
    fig_regional_series,
    load_notebook_tables,
    notebook_paths,
)

paths = notebook_paths(ROOT)
tables = load_notebook_tables(paths["final"])


## Parametri

Modificare questi valori e rilanciare le celle successive. Le variabili `PAESE` e `PAESE_CONFRONTO` sono usate per le serie territoriali; `PAESE_ISO` e `PAESE_CONFRONTO_ISO` sono usate dai grafici sui flussi dettagliati per cittadinanza.


In [ ]:
PAESE = "country:ITA"
PAESE_CONFRONTO = "country:ESP"
PAESE_ISO = "ITA"
PAESE_CONFRONTO_ISO = "ESP"

REGIONE = "region:ITC4"      # Lombardia
REGIONE_CONFRONTO = "region:ITI4"  # Lazio
PROVINCIA = "province:ITC4C" # Milano
PROVINCIA_CONFRONTO = "province:ITC11" # Torino

ANNO_FLUSSI = 2024
ANNO_NATI_ESTERO_RECENTE = 2025
ANNO_NATI_ESTERO_STORICO = 2002
ANNO_TERRITORI = 2024
ANNO_ISTRUZIONE_MIGRANTI = 2024
AREA_ISTRUZIONE_MIGRANTI = "IT"


## Saldo migratorio e componenti del bilancio

Questi grafici usano il bilancio demografico: immigrazione, emigrazione quando disponibile, saldo migratorio con aggiustamento statistico e variazione totale della popolazione. Servono per leggere il contributo delle migrazioni alla dinamica complessiva.


In [ ]:
fig_migration(tables, territory=PAESE, compare=PAESE_CONFRONTO).show()


In [ ]:
fig_population_series(tables, territory=PAESE, compare=PAESE_CONFRONTO, metric="net_migration_adjustment").show()


In [ ]:
fig_population_series(tables, territory=PAESE, compare=PAESE_CONFRONTO, metric="population_change").show()


## Et? e cittadinanza dei flussi

Eurostat pubblica i flussi annui per et?, sesso e cittadinanza. Qui la cittadinanza ? usata come misura amministrativa vicina alla nazionalit?, ma non coincide sempre con il paese di nascita. Le categorie estere sono distinte tra cittadini di altri paesi UE27 e cittadini extra UE27.


In [ ]:
fig_migration_age_profile(tables, flow="immigration", country=PAESE_ISO, compare=PAESE_CONFRONTO_ISO, year=ANNO_FLUSSI).show()


In [ ]:
fig_migration_age_profile(tables, flow="emigration", country=PAESE_ISO, compare=PAESE_CONFRONTO_ISO, year=ANNO_FLUSSI).show()


In [ ]:
fig_migration_citizenship_profile(tables, flow="immigration", country=PAESE_ISO, compare=PAESE_CONFRONTO_ISO, year=ANNO_FLUSSI).show()


In [ ]:
fig_migration_citizenship_profile(tables, flow="emigration", country=PAESE_ISO, compare=PAESE_CONFRONTO_ISO, year=ANNO_FLUSSI).show()


## Destinazioni regionali e provinciali

Il dettaglio territoriale del bilancio consente di vedere dove si concentrano immigrazione, emigrazione e saldo migratorio. A livello regionale e provinciale la fonte non contiene sempre la stessa granularit? per cittadinanza o titolo di studio: per questo la lettura territoriale resta separata dai profili dettagliati dei flussi nazionali.


In [ ]:
fig_migration_destination_rank(tables, level="region", metric="immigration", year=ANNO_TERRITORI, limit=25).show()


In [ ]:
fig_migration_destination_rank(tables, level="province", metric="immigration", year=ANNO_TERRITORI, limit=40).show()


In [ ]:
fig_regional_rank(tables, level="region", metric="net_migration_adjustment", year=ANNO_TERRITORI, limit=25).show()


In [ ]:
fig_regional_rank(tables, level="province", metric="net_migration_adjustment", year=ANNO_TERRITORI, limit=40).show()


In [ ]:
fig_regional_series(tables, focus=REGIONE, compare=REGIONE_CONFRONTO, metric="net_migration_adjustment").show()


In [ ]:
fig_regional_series(tables, focus=PROVINCIA, compare=PROVINCIA_CONFRONTO, metric="net_migration_adjustment").show()


## Titolo di studio e paese di nascita

Questa sezione usa Eurostat LFS: mostra quote percentuali della popolazione residente in famiglie private, per paese di nascita e regione NUTS2. Non ? un flusso annuo di nuovi immigrati; ? una fotografia dello stock residente. La quota di laureati ? la categoria ISCED 5-8.


In [ ]:
fig_migrant_education_by_birth(tables, geo_code=AREA_ISTRUZIONE_MIGRANTI, year=ANNO_ISTRUZIONE_MIGRANTI).show()


In [ ]:
fig_migrant_tertiary_region(tables, country=PAESE_ISO, birth_group="FOR", year=ANNO_ISTRUZIONE_MIGRANTI, limit=21).show()


In [ ]:
fig_migrant_tertiary_region(tables, country=PAESE_ISO, birth_group="NAT", year=ANNO_ISTRUZIONE_MIGRANTI, limit=21).show()


## Migrazioni interne

Se la pipeline ufficiale ISTAT ha trovato una tabella origine-destinazione, questo grafico mostra le principali destinazioni dei trasferimenti interni di residenza. Se la tabella non ? presente, la cella resta vuota ma il notebook continua a funzionare.


In [ ]:
fig_internal_migration_destinations(tables, year=ANNO_TERRITORI, limit=40).show()


## Popolazione nata all'estero

Il Kebab demografico qui usa lo stock dei residenti nati all'estero per et? e sesso. ? utile per confrontare la struttura anagrafica della popolazione immigrata con quella complessiva, ma non va letto come flusso annuale.


In [ ]:
fig_kebab(tables, territory=PAESE, year=ANNO_NATI_ESTERO_STORICO, population_kind="foreign_born").show()


In [ ]:
fig_kebab(tables, territory=PAESE, year=ANNO_NATI_ESTERO_RECENTE, population_kind="foreign_born").show()
